# Colab CPU setup for this project

This notebook serves the current project folder from a Colab CPU environment so you can browse it over HTTP.

In [ ]:
import os
import threading
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print('Google Colab detected. Mounting Google Drive...')
    drive.mount('/content/drive')
else:
    print('Running outside Colab. The current workspace will be used if it contains index.html.')

candidates = []
for base in [Path('/content'), Path('/content/drive/MyDrive'), Path.cwd()]:
    if not base.exists():
        continue
    candidates.extend([
        base / '2ndbrain-v3',
        base / '2ndbrain-v3 orginal',
        base / '2ndbrain',
        base
    ])

project_dir = None
for path in candidates:
    if path.exists() and (path / 'index.html').exists():
        project_dir = path
        break

if project_dir is None:
    raise FileNotFoundError('Project folder not found automatically. Upload it to Colab or place it in /content/drive/MyDrive/2ndbrain-v3.')

project_dir = project_dir.resolve()
print(f'Using project directory: {project_dir}')

In [ ]:
port = 8000

def start_server(folder: str, port: int = 8000) -> None:
    os.chdir(folder)
    handler = partial(SimpleHTTPRequestHandler, directory=folder)
    server = ThreadingHTTPServer(('0.0.0.0', port), handler)
    print(f'Serving {folder} on http://127.0.0.1:{port}')
    print('Open the address in a browser tab to view the project.')
    server.serve_forever()

thread = threading.Thread(target=start_server, args=[str(project_dir), port], daemon=True)
thread.start()
print('Server started in the background.')